Povoamento de Dados- Tabela de Factos Desempenho Instrutor

Importação de Pacotes

In [10]:
import pandas as pd
import numpy as np
from datetime import datetime

Extração de Informação necessária

In [11]:
dim_instrutor = pd.read_csv("../Dados Finais/dim_instrutor.csv",  encoding="utf-8-sig")
dim_treino = pd.read_csv("../Dados Finais/dim_treino.csv",  encoding="utf-8-sig")
dim_aula = pd.read_csv("../Dados Finais/dim_aula.csv",  encoding="utf-8-sig")

idas_ginasio = pd.read_csv('../Fontes/idas_ginasio_set.csv', encoding='latin1')

idas_ginasio.head()

,data,id_cliente,id_treino,id_aula,id_campanha,hora_entrada,hora_saida,avaliacao_treino,avaliacao_aula,semana,dia_semana
0,2025-09-01,14,14.0,1.0,C004,18,20,5.0,3.0,2025-09-01,0
1,2025-09-01,78,20.0,1.0,C003,19,20,5.0,NaN,2025-09-01,0
2,2025-09-01,35,NaN,2.0,C007,19,22,NaN,4.0,2025-09-01,0
3,2025-09-01,59,16.0,1.0,NaN,15,17,NaN,1.0,2025-09-01,0
4,2025-09-01,52,12.0,NaN,NaN,15,18,1.0,NaN,2025-09-01,0


Verificação da Integridade Referencial

In [12]:
idas_ginasio['id_aula'] = idas_ginasio['id_aula'].astype('Int64')
idas_ginasio['id_treino'] = idas_ginasio['id_treino'].astype('Int64')

# Aulas
aulas_faltam = set(idas_ginasio['id_aula'].dropna()) - set(dim_aula['aula_id'])
# Treinos
treinos_faltam = set(idas_ginasio['id_treino'].dropna()) - set(dim_treino['treino_id_atividade'])

# Instrutores (via treino)
instrutores_treino = set(dim_treino['instrutor_nome'])
instrutores_dim = set(dim_instrutor['instrutor_nome'])
instrutores_faltam = instrutores_treino - instrutores_dim

# Resumo
aulas_faltam, treinos_faltam, instrutores_faltam

(set(), set(), set())

Atribuição de Instrutores às Aulas com base nas suas Especializações

In [13]:
# Mapear especialização do instrutor para nome da aula (por keywords)
mapa_especializacao = {
    'Musculação': ['musculação'],
    'Yoga': ['yoga'],
    'Pilates': ['pilates'],
    'Cardio': ['cardio'],
    'Crossfit': ['crossfit'],
    'Boxe': ['boxe'],
    'Alongamentos': ['alongamento', 'alongamentos'],
    'Funcional': ['funcional'],
    'HIIT': ['hiit'],
    'Zumba': ['zumba']
}

# Atribuir cada aula a um instrutor com base na especialização
atribuicoes = []
restantes_instrutores = set(dim_instrutor['instrutor_nome'])
restantes_aulas = set(dim_aula['aula_id'])

for idx, aula in dim_aula.iterrows():
    aula_nome = aula['aula_nome'].lower()
    instrutor_encontrado = None
    for idx2, instrutor in dim_instrutor.iterrows():
        area = instrutor['instrutor_especializacao']
        if area in mapa_especializacao:
            if any(kw in aula_nome for kw in mapa_especializacao[area]):
                instrutor_encontrado = instrutor['instrutor_nome']
                break
    if instrutor_encontrado:
        atribuicoes.append((aula['aula_id'], aula['aula_nome'], instrutor_encontrado))
        restantes_instrutores.discard(instrutor_encontrado)
        restantes_aulas.discard(aula['aula_id'])

# Se sobrarem aulas/instrutores, atribuir aleatoriamente
import random
for aula_id in restantes_aulas:
    aula_nome = dim_aula.loc[dim_aula['aula_id'] == aula_id, 'aula_nome'].values[0]
    instrutor_nome = restantes_instrutores.pop()
    atribuicoes.append((aula_id, aula_nome, instrutor_nome))

# Mostrar atribuições finais
atribuicoes

[(1, 'Aula de Alongamentos', 'Kevin Araújo Simões'),
 (2, 'Aula de Boxe', 'Ivan Carneiro'),
 (3, 'Aula de Cardio', 'Sara Neto'),
 (4, 'Aula de Crossfit', 'Leandro Amaral'),
 (5, 'Aula de Funcional', 'Marcos Castro Carneiro'),
 (6, 'Aula de HIIT', 'Mário Jesus'),
 (7, 'Aula de Musculação', 'Enzo Pacheco'),
 (8, 'Aula de Pilates', 'Sérgio Pires'),
 (9, 'Aula de Yoga', 'Anita Antunes'),
 (10, 'Aula de Zumba', 'Pilar Cruz')]

Preparar as Datas das Semanas (primeiro dia de cada semana do mês de Setembro 2025)

In [14]:
idas_ginasio['data'] = pd.to_datetime(idas_ginasio['data'])
idas_ginasio['semana'] = idas_ginasio['data'].dt.to_period('W').apply(lambda r: r.start_time)
semanas = sorted(idas_ginasio['semana'].unique())

Preparar surrogate keys para joins

In [15]:
instrutor_nome_to_sk = dict(zip(dim_instrutor['instrutor_nome'], dim_instrutor['instrutor_sk']))
aula_id_to_sk = dict(zip(dim_aula['aula_id'], dim_aula['aula_sk']))

Gerar Registos Semanais

In [16]:
mapa_aula_instrutor = {
    1: 'Kevin Araújo Simões',
    2: 'Ivan Carneiro',
    3: 'Sara Neto',
    4: 'Leandro Amaral',
    5: 'Marcos Castro Carneiro',
    6: 'Mário Jesus',
    7: 'Enzo Pacheco',
    8: 'Sérgio Pires',
    9: 'Anita Antunes',
    10: 'Pilar Cruz'
}

registos = []
for semana in semanas:
    for aula_id, instrutor_nome in mapa_aula_instrutor.items():
        sk_instrutor = instrutor_nome_to_sk[instrutor_nome]
        sk_aula = aula_id_to_sk[aula_id]
        # Filtrar idas à aula nesta semana
        idas_aula = idas_ginasio[(idas_ginasio['id_aula'] == aula_id) & (idas_ginasio['semana'] == semana)]
        # Lotação: nº presenças / capacidade (uma sessão por semana)
        presencas = len(idas_aula)
        capacidade = int(dim_aula.loc[dim_aula['aula_id'] == aula_id, 'aula_capacidade'].values[0])
        lotacao = presencas / capacidade if capacidade > 0 else None
        # Avaliação média das aulas
        avaliacoes_aula = idas_aula['avaliacao_aula'].dropna()
        avaliacao_media_aula = avaliacoes_aula.mean() if not avaliacoes_aula.empty else None
        # Treinos do instrutor
        treinos_instrutor = dim_treino[dim_treino['instrutor_nome'] == instrutor_nome]['treino_id_atividade'].tolist()
        idas_treino = idas_ginasio[(idas_ginasio['id_treino'].isin(treinos_instrutor)) & (idas_ginasio['semana'] == semana)]
        n_planos_seguidos = len(idas_treino)
        avaliacoes_treino = idas_treino['avaliacao_treino'].dropna()
        avaliacao_media_treino = avaliacoes_treino.mean() if not avaliacoes_treino.empty else None
        registos.append({
            'sk_instrutor': sk_instrutor,
            'sk_aula': sk_aula,
            'semana_data': semana,
            'lotacao_aula': lotacao,
            'avaliacao_media_aula': avaliacao_media_aula,
            'n_planos_seguidos': n_planos_seguidos,
            'avaliacao_media_treino': avaliacao_media_treino
        })

Adicionar ID e ordenar colunas

In [17]:
fact_instrutor = pd.DataFrame(registos)
fact_instrutor = fact_instrutor.reset_index(drop=True)
fact_instrutor['fact_id'] = fact_instrutor.index + 1
col_order = ['fact_id', 'sk_instrutor', 'sk_aula', 'semana_data', 'lotacao_aula',
             'avaliacao_media_aula', 'n_planos_seguidos', 'avaliacao_media_treino']
fact_instrutor = fact_instrutor[col_order]

# Amostra
fact_instrutor

,fact_id,sk_instrutor,sk_aula,semana_data,lotacao_aula,avaliacao_media_aula,n_planos_seguidos,avaliacao_media_treino
0,1,9,1,2025-09-01,0.375,2.333333,8,3.666667
1,2,7,2,2025-09-01,0.325,4.000000,20,2.875000
2,3,4,3,2025-09-01,0.425,2.400000,23,2.333333
3,4,6,4,2025-09-01,0.375,3.000000,20,3.250000
4,5,8,5,2025-09-01,0.400,3.142857,22,3.555556
5,6,10,6,2025-09-01,0.375,4.333333,6,3.000000
6,7,1,7,2025-09-01,0.325,3.333333,30,3.555556
7,8,5,8,2025-09-01,0.300,3.400000,23,3.750000
8,9,2,9,2025-09-01,0.225,2.000000,5,3.000000
9,10,3,10,2025-09-01,0.325,2.600000,29,2.700000


In [18]:
fact_instrutor.to_csv("../Dados Finais/tf_instrutor.csv", index=False, encoding="utf-8-sig")
print("Tabela de Factos Desempenho Instrutor pronta para carga!")
display(fact_instrutor.head())

Tabela de Factos Desempenho Instrutor pronta para carga!


,fact_id,sk_instrutor,sk_aula,semana_data,lotacao_aula,avaliacao_media_aula,n_planos_seguidos,avaliacao_media_treino
0,1,9,1,2025-09-01,0.375,2.333333,8,3.666667
1,2,7,2,2025-09-01,0.325,4.000000,20,2.875000
2,3,4,3,2025-09-01,0.425,2.400000,23,2.333333
3,4,6,4,2025-09-01,0.375,3.000000,20,3.250000
4,5,8,5,2025-09-01,0.400,3.142857,22,3.555556
